In [1]:
import joblib
import pandas as pd

# Load the saved model
loaded_model = joblib.load("model.joblib")
print("✅ Model loaded successfully!")

✅ Model loaded successfully!


In [44]:
import os
import shutil

# Create the models directory
os.makedirs("models", exist_ok=True)

# Copy your existing model file into the models/ folder
if os.path.exists("model.joblib"):
    shutil.copy("model.joblib", "models/model.joblib")
    print("✅ Successfully moved model to models/model.joblib!")
else:
    print("⚠️ 'model.joblib' not found in current directory!")

✅ Successfully moved model to models/model.joblib!


In [40]:
%%writefile app/services/model_service.py
import joblib
import os

# Safe loader that checks both root and models directory
model_path = "models/model.joblib" if os.path.exists("models/model.joblib") else "model.joblib"

try:
    model = joblib.load(model_path)
except Exception as e:
    model = None

def predict(features: list):
    if model is None:
        return 0
    return int(model.predict([features])[0])

Overwriting app/services/model_service.py


In [3]:
import os

folders = [
    "app/core",
    "app/api",
    "app/services",
    "app/middleware",
    "models",
    "monitoring",
]

for f in folders:
    os.makedirs(f, exist_ok=True)

for pkg in [
    "app",
    "app/core",
    "app/api",
    "app/services",
    "app/middleware",
]:
    open(os.path.join(pkg, "__init__.py"), "a").close()

print("✅ Structure created!")

✅ Structure created!


In [4]:
%%bash
git init
echo "__pycache__/\n*.pyc\n*.pyo\n*.pyd\n*.db\n.env\nmodels/model.joblib\n.ipynb_checkpoints/" > .gitignore
git add .
git commit -m "Initial commit"
git branch -M main
git remote add origin https://github.com/<your-username>/<PROJECT-NAME>.git
git push -u origin main


Reinitialized existing Git repository in C:/Users/AMAN KUMAR VERMA/Downloads/fastapi_projects/.git/


[main 9c6c68b] Initial commit
 7 files changed, 661 insertions(+), 129 deletions(-)


bash: line 6: your-username: No such file or directory
To https://github.com/amankumarverma2703akv-dot/fastapi_projects.git
   ac512b9..9c6c68b  main -> main


branch 'main' set up to track 'origin/main'.


In [5]:
%%writefile app/core/config.py
from pydantic_settings import BaseSettings

class Settings(BaseSettings):
    PROJECT_NAME: str = "BFSI1"
    SECRET_KEY: str = "super-secret-key-123"
    ALGORITHM: str = "HS256"
    ACCESS_TOKEN_EXPIRE_MINUTES: int = 30
    REDIS_URL: str = "redis://redis:6379"

    class Config:
        env_file = ".env"

settings = Settings()

Overwriting app/core/config.py


In [6]:
%%writefile app/core/security.py
from datetime import datetime, timedelta
from jose import jwt, JWTError
from app.core.config import settings

def create_access_token(data: dict, expires_delta: timedelta = None):
    to_encode = data.copy()
    expire = datetime.utcnow() + (expires_delta or timedelta(minutes=settings.ACCESS_TOKEN_EXPIRE_MINUTES))
    to_encode.update({"exp": expire})
    return jwt.encode(to_encode, settings.SECRET_KEY, algorithm=settings.ALGORITHM)

def verify_token(token: str):
    try:
        payload = jwt.decode(token, settings.SECRET_KEY, algorithms=[settings.ALGORITHM])
        return payload
    except JWTError:
        return None


Overwriting app/core/security.py


In [7]:
%%writefile app/api/dependencies.py
from fastapi import Depends, HTTPException, status
from fastapi.security import OAuth2PasswordBearer
from app.core.security import verify_token

oauth2_scheme = OAuth2PasswordBearer(tokenUrl="login")

def get_current_user(token: str = Depends(oauth2_scheme)):
    payload = verify_token(token)
    if payload is None:
        raise HTTPException(status_code=status.HTTP_401_UNAUTHORIZED, detail="Invalid token")
    return payload


Overwriting app/api/dependencies.py


In [8]:
%%writefile app/api/exceptions.py
from fastapi.responses import JSONResponse
from fastapi import Request

async def http_exception_handler(request: Request, exc):
    return JSONResponse(status_code=exc.status_code, content={"detail": exc.detail})


Overwriting app/api/exceptions.py


In [9]:
%%writefile app/api/routes_auth.py
from fastapi import APIRouter, Depends
from fastapi.security import OAuth2PasswordRequestForm
from app.core.security import create_access_token

router = APIRouter()

@router.post("/login")
def login(form_data: OAuth2PasswordRequestForm = Depends()):
    if form_data.username != "admin" or form_data.password != "password":
        return {"error": "Invalid credentials"}
    token = create_access_token({"sub": form_data.username})
    return {"access_token": token, "token_type": "bearer"}


Overwriting app/api/routes_auth.py


In [10]:
%%writefile app/services/redis_cache.py
import redis
from app.core.config import settings

redis_client = redis.Redis.from_url(settings.REDIS_URL, decode_responses=True)

def get_cache(key: str):
    return redis_client.get(key)

def set_cache(key: str, value: str, expire: int = 300):
    redis_client.set(key, value, ex=expire)


Overwriting app/services/redis_cache.py


In [11]:
%%writefile app/services/model_service.py
import joblib

model, scaler = joblib.load("models/model.joblib")

def predict(features: list):
    scaled = scaler.transform([features])
    return int(model.predict(scaled)[0])


Overwriting app/services/model_service.py


In [12]:
%%writefile app/api/routes_predict.py
from fastapi import APIRouter, Depends
from app.api.dependencies import get_current_user
from app.services.redis_cache import get_cache, set_cache
from app.services.model_service import predict

router = APIRouter()

@router.post("/predict")
def predict_endpoint(features: list, user: dict = Depends(get_current_user)):
    key = str(features)
    cached = get_cache(key)
    if cached:
        return {"prediction": int(cached), "cached": True}
    result = predict(features)
    set_cache(key, result)
    return {"prediction": result, "cached": False}


Overwriting app/api/routes_predict.py


In [13]:
%%writefile app/middleware/logging_middleware.py
from starlette.middleware.base import BaseHTTPMiddleware
import logging

logger = logging.getLogger("uvicorn")

class LoggingMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request, call_next):
        logger.info(f"Request: {request.method} {request.url}")
        response = await call_next(request)
        logger.info(f"Response status: {response.status_code}")
        return response


Overwriting app/middleware/logging_middleware.py


In [14]:
%%writefile app/main.py
from fastapi import FastAPI
from app.api import routes_auth, routes_predict
from app.middleware.logging_middleware import LoggingMiddleware

app = FastAPI()
app.include_router(routes_auth.router)
app.include_router(routes_predict.router)
app.add_middleware(LoggingMiddleware)


Overwriting app/main.py


In [28]:
import nest_asyncio
import uvicorn

nest_asyncio.apply()
uvicorn.run("app.main:app", host="0.0.0.0", port=8000, reload=True)


INFO:     Will watch for changes in these directories: ['c:\\Users\\AMAN KUMAR VERMA\\Downloads\\fastapi_projects']
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [30764] using StatReload
INFO:     Stopping reloader process [30764]


In [24]:
%%writefile monitoring/prometheus.yml
global:
  scrape_interval: 15s
scrape_configs:
  - job_name: 'fastapi'
    static_configs:
      - targets: ['app:8000']


Overwriting monitoring/prometheus.yml


In [25]:
%%writefile docker-compose.yml
services:
  app:
    build:
      context: .
      dockerfile: docker/Dockerfile
    ports:
      - "8000:8000"
    environment:
      - REDIS_URL=redis://redis:6379
      - SECRET_KEY=mysecretkey
    depends_on:
      - redis

  redis:
    image: redis:alpine
    ports:
      - "6379:6379"

Overwriting docker-compose.yml


In [26]:
import os

os.makedirs("docker", exist_ok=True)
print("✅ 'docker' directory ready!")

✅ 'docker' directory ready!


In [42]:
%%writefile docker/Dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8000

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]

Overwriting docker/Dockerfile


In [43]:
%%writefile requirements.txt
fastapi
uvicorn
python-jose
pydantic
pydantic-settings
redis
scikit-learn
joblib
prometheus-client
python-multipart

Overwriting requirements.txt


In [21]:
! docker ps - a

docker: 'docker ps' accepts no arguments

Usage:  docker ps [OPTIONS]

Run 'docker ps --help' for more information


In [ ]:
! docker compose up -d --build

unable to get image 'redis:alpine': failed to connect to the docker API at npipe:////./pipe/dockerDesktopLinuxEngine; check if the path is correct and if the daemon is running: open //./pipe/dockerDesktopLinuxEngine: The system cannot find the file specified.


In [41]:
%%writefile render.yaml
services:
  - type: web
    name: fastapi-ml-app
    env: python
    buildCommand: "pip install -r requirements.txt"
    startCommand: "uvicorn app.main:app --host 0.0.0.0 --port 8000"
envVars:
  - key: REDIS_URL
    value: "redis://<your-free-redis-instance>"


Overwriting render.yaml


In [19]:
! docker compose logs app

app-1  | Traceback (most recent call last):
app-1  |   File "/usr/local/bin/uvicorn", line 8, in <module>
app-1  |     sys.exit(main())
app-1  |              ^^^^^^
app-1  |   File "/usr/local/lib/python3.11/site-packages/click/core.py", line 1569, in __call__
app-1  |     return self.main(*args, **kwargs)
app-1  |            ^^^^^^^^^^^^^^^^^^^^^^^^^^
app-1  |   File "/usr/local/lib/python3.11/site-packages/click/core.py", line 1490, in main
app-1  |     rv = self.invoke(ctx)
app-1  |          ^^^^^^^^^^^^^^^^
app-1  |   File "/usr/local/lib/python3.11/site-packages/click/core.py", line 1353, in invoke
app-1  |     return ctx.invoke(self.callback, **ctx.params)
app-1  |            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
app-1  |   File "/usr/local/lib/python3.11/site-packages/click/core.py", line 907, in invoke
app-1  |     return callback(*args, **kwargs)
app-1  |            ^^^^^^^^^^^^^^^^^^^^^^^^^
app-1  |   File "/usr/local/lib/python3.11/site-packages/uvicorn/main.py", line 440,

In [17]:
# 1. Wipe old crashed container state
! docker compose down

# 2. Rebuild and launch
! docker compose up -d --build

# 3. Check status
! docker ps

 Container fastapi_projects-app-1 Stopping 
 Container fastapi_projects-app-1 Stopped 
 Container fastapi_projects-app-1 Removing 
 Container fastapi_projects-app-1 Removed 
 Container fastapi_projects-redis-1 Stopping 
 Container fastapi_projects-redis-1 Stopped 
 Container fastapi_projects-redis-1 Removing 
 Container fastapi_projects-redis-1 Removed 
 Network fastapi_projects_default Removing 
 Network fastapi_projects_default Removed 


#1 [internal] load local bake definitions
#1 reading from stdin 579B done
#1 DONE 0.0s

#2 [internal] load build definition from Dockerfile
#2 transferring dockerfile: 258B 0.0s done
#2 DONE 0.0s

#3 [internal] load metadata for docker.io/library/python:3.11-slim
#3 ...

#4 [auth] library/python:pull token for registry-1.docker.io
#4 DONE 0.0s

#3 [internal] load metadata for docker.io/library/python:3.11-slim
#3 DONE 6.1s

#5 [internal] load .dockerignore
#5 transferring context: 2B done
#5 DONE 0.0s

#6 [1/5] FROM docker.io/library/python:3.11-slim@sha256:90744cff8f32887f075c47d747a173ff333e9e98801667af93c357fa9f5e28ff
#6 resolve docker.io/library/python:3.11-slim@sha256:90744cff8f32887f075c47d747a173ff333e9e98801667af93c357fa9f5e28ff 0.1s done
#6 DONE 0.1s

#7 [internal] load build context
#7 transferring context: 139.95kB 0.1s done
#7 DONE 0.1s

#8 [2/5] WORKDIR /app
#8 CACHED

#9 [3/5] COPY requirements.txt .
#9 CACHED

#10 [4/5] RUN pip install --no-cache-dir -r requirements.txt


 Image fastapi_projects-app Building 
 Image fastapi_projects-app Built 
 Network fastapi_projects_default Creating 
 Network fastapi_projects_default Created 
 Container fastapi_projects-redis-1 Creating 
 Container fastapi_projects-redis-1 Created 
 Container fastapi_projects-app-1 Creating 
 Container fastapi_projects-app-1 Created 
 Container fastapi_projects-redis-1 Starting 
 Container fastapi_projects-redis-1 Started 
 Container fastapi_projects-app-1 Starting 
 Container fastapi_projects-app-1 Started 


CONTAINER ID   IMAGE                  COMMAND                  CREATED        STATUS                  PORTS                                         NAMES
f3628461d8be   fastapi_projects-app   "uvicorn app.main:ap…"   1 second ago   Up Less than a second   0.0.0.0:8000->8000/tcp, [::]:8000->8000/tcp   fastapi_projects-app-1
a05729d46d64   redis:alpine           "docker-entrypoint.s…"   1 second ago   Up 1 second             0.0.0.0:6379->6379/tcp, [::]:6379->6379/tcp   fastapi_projects-redis-1


# restart docker

In [37]:
# 1. Check if containers are 'Up' or 'Exited'
! docker ps -a

# 2. View the actual error log inside the app container
! docker compose logs app

CONTAINER ID   IMAGE                             COMMAND                  CREATED         STATUS                      PORTS                                         NAMES
1f0d0eb759e1   fastapi_projects-app              "uvicorn app.main:ap…"   2 minutes ago   Exited (1) 2 minutes ago                                                  fastapi_projects-app-1
a05729d46d64   redis:alpine                      "docker-entrypoint.s…"   7 minutes ago   Up 7 minutes                0.0.0.0:6379->6379/tcp, [::]:6379->6379/tcp   fastapi_projects-redis-1
9781a08f49df   fastapi-fastapi_app               "uvicorn main:app --…"   2 days ago      Exited (255) 25 hours ago   0.0.0.0:8000->8000/tcp                        fastapi_app
50b3e12db86f   grafana/grafana:latest            "/run.sh"                2 days ago      Exited (255) 25 hours ago   0.0.0.0:3000->3000/tcp                        grafana_dashboard
e962503af60e   prom/prometheus:v2.45.0           "/bin/prometheus --c…"   3 days ago      Exited

In [30]:
! docker compose up -d --build

#1 [internal] load local bake definitions
#1 reading from stdin 579B done
#1 DONE 0.0s

#2 [internal] load build definition from Dockerfile
#2 transferring dockerfile: 258B 0.0s done
#2 DONE 0.0s

#3 [internal] load metadata for docker.io/library/python:3.11-slim
#3 DONE 1.1s

#4 [internal] load .dockerignore
#4 transferring context: 2B done
#4 DONE 0.0s

#5 [1/5] FROM docker.io/library/python:3.11-slim@sha256:90744cff8f32887f075c47d747a173ff333e9e98801667af93c357fa9f5e28ff
#5 resolve docker.io/library/python:3.11-slim@sha256:90744cff8f32887f075c47d747a173ff333e9e98801667af93c357fa9f5e28ff 0.1s done
#5 DONE 0.1s

#6 [internal] load build context
#6 transferring context: 89.08kB 0.1s done
#6 DONE 0.1s

#7 [2/5] WORKDIR /app
#7 CACHED

#8 [3/5] COPY requirements.txt .
#8 CACHED

#9 [4/5] RUN pip install --no-cache-dir -r requirements.txt
#9 CACHED

#10 [5/5] COPY . .
#10 DONE 0.1s

#11 exporting to image
#11 exporting layers
#11 exporting layers 1.7s done
#11 exporting manifest sha256:f1

 Image fastapi_projects-app Building 
 Image fastapi_projects-app Built 
 Container fastapi_projects-redis-1 Running 
 Container fastapi_projects-app-1 Recreate 
 Container fastapi_projects-app-1 Recreated 
 Container fastapi_projects-app-1 Starting 
 Container fastapi_projects-app-1 Started 


In [38]:
# Rebuild image with python-multipart installed
! docker compose up -d --build

#1 [internal] load local bake definitions
#1 reading from stdin 579B done
#1 DONE 0.0s

#2 [internal] load build definition from Dockerfile
#2 transferring dockerfile: 258B 0.0s done
#2 DONE 0.0s

#3 [internal] load metadata for docker.io/library/python:3.11-slim
#3 DONE 1.1s

#4 [internal] load .dockerignore
#4 transferring context: 2B done
#4 DONE 0.0s

#5 [1/5] FROM docker.io/library/python:3.11-slim@sha256:90744cff8f32887f075c47d747a173ff333e9e98801667af93c357fa9f5e28ff
#5 resolve docker.io/library/python:3.11-slim@sha256:90744cff8f32887f075c47d747a173ff333e9e98801667af93c357fa9f5e28ff 0.0s done
#5 DONE 0.1s

#6 [internal] load build context
#6 transferring context: 105.20kB 0.1s done
#6 DONE 0.1s

#7 [2/5] WORKDIR /app
#7 CACHED

#8 [3/5] COPY requirements.txt .
#8 CACHED

#9 [4/5] RUN pip install --no-cache-dir -r requirements.txt
#9 CACHED

#10 [5/5] COPY . .
#10 DONE 0.2s

#11 exporting to image
#11 exporting layers
#11 exporting layers 1.7s done
#11 exporting manifest sha256:3

 Image fastapi_projects-app Building 
 Image fastapi_projects-app Built 
 Container fastapi_projects-redis-1 Running 
 Container fastapi_projects-app-1 Recreate 
 Container fastapi_projects-app-1 Recreated 
 Container fastapi_projects-app-1 Starting 
 Container fastapi_projects-app-1 Started 


In [39]:
import time
import requests

time.sleep(3)  # Give Uvicorn a few seconds to start
login_data = {"username": "admin", "password": "password"}
res = requests.post("http://127.0.0.1:8000/login", data=login_data)
print("🔑 Login Output:", res.json())

ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /login (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=8000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))

In [35]:
import requests

# 1. Test Healthcheck
res = requests.get("http://127.0.0.1:8000/")
print("✅ Healthcheck Output:", res.json())

# 2. Test Login
login_data = {"username": "admin", "password": "password"}
auth_res = requests.post("http://127.0.0.1:8000/login", data=login_data)
print("🔑 Login Output:", auth_res.json())

ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: / (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=8000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))

In [ ]:
%%writefile app/main.py
from fastapi import FastAPI
from app.api import routes_auth, routes_predict

app = FastAPI(title="BFSI Loan Approval API")

@app.get("/")
def root():
    return {"status": "ok", "message": "BFSI Loan Approval API is running!"}

app.include_router(routes_auth.router)
app.include_router(routes_predict.router)

Overwriting app/main.py
